In [0]:
# Cell 1: Config & Imports

dbutils.widgets.text("input_folder", "Hackathon2/", "Input DICOM Folder")
dbutils.widgets.text("deployment_name", "gpt-4.1-mini-b", "Model Deployment Name")
dbutils.widgets.text("poll_interval_seconds", "60", "Batch Poll Interval (seconds)")

import pydicom
import numpy as np
from PIL import Image
from openai import OpenAI
from io import BytesIO
import base64
import json
import time
import datetime
import tempfile
import os

api_key = dbutils.secrets.get(scope="adc_store", key="hackathon_gpt41mini_api_key")
endpoint = "https://telefonica-hackathon-2026.cognitiveservices.azure.com/openai/v1/"
deployment_name = dbutils.widgets.get("deployment_name")
poll_interval = int(dbutils.widgets.get("poll_interval_seconds"))

client = OpenAI(base_url=endpoint, api_key=api_key)

print(f"Endpoint: {endpoint}")
print(f"Deployment: {deployment_name}")
print(f"Poll interval: {poll_interval}s")

In [0]:
# Cell 2: List DICOM Files

input_dir = f"/Volumes/1_inland/sectra/vone/{dbutils.widgets.get('input_folder')}"

df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .option("includeBinaryFiles", "false")
    .load(input_dir)
    .select("path")
)

dicom_paths = [row.path for row in df.collect()]
print(f"Found {len(dicom_paths)} DICOM files")

In [0]:
# Cell 3: Generate Batch JSONL

SYSTEM_PROMPT = (
    "You are a medical image PII detector. Analyze the image and determine: "
    "(1) whether any text is visible, "
    "(2) whether that text contains personal information "
    "(e.g. patient name, date of birth, scan date, ID numbers, address). "
    "Return confidence_score as a float from 0.0 to 1.0 where 0.0 means certainly no PII "
    "and 1.0 means certainly PII present. "
    "If no text is detected, set confidence_score to 0.0 and detected_text_summary to an empty string."
)

USER_PROMPT = "Analyze this medical image for visible text and personal information."

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "PIIDetectionResult",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "is_text_detected": {"type": "boolean"},
                "is_personal_information_detected": {"type": "boolean"},
                "confidence_score": {"type": "number"},
                "detected_text_summary": {"type": "string"},
            },
            "required": [
                "is_text_detected",
                "is_personal_information_detected",
                "confidence_score",
                "detected_text_summary",
            ],
            "additionalProperties": False,
        },
    },
}

# Map custom_id -> path (custom_id may have length limits, so use index)
id_to_path = {}
dicom_read_failures = []

jsonl_path = tempfile.mktemp(suffix=".jsonl")

with open(jsonl_path, "w") as f:
    for idx, path in enumerate(dicom_paths):
        custom_id = f"task-{idx}"
        local_path = path[5:] if path.startswith("dbfs:") else path
        id_to_path[custom_id] = path

        try:
            ds = pydicom.dcmread(local_path, stop_before_pixels=False)
            arr = ds.pixel_array

            arr_min, arr_max = arr.min(), arr.max()
            if arr_max > arr_min:
                arr = ((arr - arr_min) * (255.0 / (arr_max - arr_min))).astype(np.uint8)
            else:
                arr = np.zeros(arr.shape, dtype=np.uint8)

            image = Image.fromarray(arr)
            buffer = BytesIO()
            image.save(buffer, format="PNG", optimize=True)
            base64_str = base64.b64encode(buffer.getbuffer()).decode("utf-8")
            buffer.close()
            del arr, image

            line = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": deployment_name,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": USER_PROMPT},
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": f"data:image/png;base64,{base64_str}",
                                        "detail": "low",
                                    },
                                },
                            ],
                        },
                    ],
                    "max_tokens": 200,
                    "response_format": RESPONSE_FORMAT,
                },
            }
            f.write(json.dumps(line) + "\n")

        except Exception as e:
            dicom_read_failures.append({"path": path, "error": str(e)[:1000]})

jsonl_size_mb = os.path.getsize(jsonl_path) / (1024 * 1024)
batch_request_count = len(id_to_path) - len(dicom_read_failures)
print(f"JSONL generated: {jsonl_path}")
print(f"  Batch requests: {batch_request_count}")
print(f"  DICOM read failures: {len(dicom_read_failures)}")
print(f"  File size: {jsonl_size_mb:.1f} MB")

In [0]:
# Cell 4: Upload File & Submit Batch Job

file_response = client.files.create(
    file=open(jsonl_path, "rb"),
    purpose="batch",
    extra_body={"expires_after": {"seconds": 1209600, "anchor": "created_at"}},
)
file_id = file_response.id
print(f"Uploaded file: {file_id} (status: {file_response.status})")

batch_response = client.batches.create(
    input_file_id=file_id,
    endpoint="/chat/completions",
    completion_window="24h",
)
batch_id = batch_response.id
print(f"Batch job created: {batch_id} (status: {batch_response.status})")

# Clean up local temp file
os.remove(jsonl_path)

In [0]:
# Cell 5: Poll for Completion

status = batch_response.status
while status not in ("completed", "failed", "cancelled"):
    time.sleep(poll_interval)
    batch_response = client.batches.retrieve(batch_id)
    status = batch_response.status
    counts = batch_response.request_counts
    print(
        f"{datetime.datetime.now():%H:%M:%S} | "
        f"Status: {status} | "
        f"Completed: {counts.completed}/{counts.total} | "
        f"Failed: {counts.failed}"
    )

if status == "failed":
    print("BATCH FAILED:")
    for error in batch_response.errors.data:
        print(f"  [{error.code}] {error.message}")
elif status == "cancelled":
    print("Batch was cancelled.")
else:
    print(f"Batch completed. Output file: {batch_response.output_file_id}")

In [0]:
# Cell 6: Download & Parse Results

results = []

# Parse successful outputs
output_file_id = batch_response.output_file_id
if output_file_id:
    output_content = client.files.content(output_file_id).text
    for line in output_content.strip().split("\n"):
        resp = json.loads(line)
        custom_id = resp["custom_id"]
        path = id_to_path.get(custom_id, custom_id)

        if resp.get("error"):
            results.append({
                "input_path": path,
                "is_loaded_ok": True,
                "is_text_detected": None,
                "is_pii_detected": None,
                "confidence_score": None,
                "detected_text_summary": None,
                "llm_raw_result": None,
                "error_message": json.dumps(resp["error"]),
            })
        else:
            try:
                raw_content = resp["response"]["body"]["choices"][0]["message"]["content"]
                parsed = json.loads(raw_content)
                results.append({
                    "input_path": path,
                    "is_loaded_ok": True,
                    "is_text_detected": parsed["is_text_detected"],
                    "is_pii_detected": parsed["is_personal_information_detected"],
                    "confidence_score": float(parsed["confidence_score"]),
                    "detected_text_summary": parsed["detected_text_summary"],
                    "llm_raw_result": raw_content,
                    "error_message": None,
                })
            except Exception as e:
                results.append({
                    "input_path": path,
                    "is_loaded_ok": True,
                    "is_text_detected": None,
                    "is_pii_detected": None,
                    "confidence_score": None,
                    "detected_text_summary": None,
                    "llm_raw_result": str(resp.get("response", "")),
                    "error_message": f"Parse error: {str(e)[:500]}",
                })

# Parse error file if present
error_file_id = batch_response.error_file_id
if error_file_id:
    error_content = client.files.content(error_file_id).text
    for line in error_content.strip().split("\n"):
        if not line:
            continue
        resp = json.loads(line)
        custom_id = resp["custom_id"]
        path = id_to_path.get(custom_id, custom_id)
        # Only add if not already in results (avoid duplicates)
        if not any(r["input_path"] == path for r in results):
            results.append({
                "input_path": path,
                "is_loaded_ok": True,
                "is_text_detected": None,
                "is_pii_detected": None,
                "confidence_score": None,
                "detected_text_summary": None,
                "llm_raw_result": None,
                "error_message": json.dumps(resp.get("error", resp)),
            })

# Add DICOM read failures
for fail in dicom_read_failures:
    results.append({
        "input_path": fail["path"],
        "is_loaded_ok": False,
        "is_text_detected": None,
        "is_pii_detected": None,
        "confidence_score": None,
        "detected_text_summary": None,
        "llm_raw_result": None,
        "error_message": fail["error"],
    })

print(f"Total results: {len(results)}")
print(f"  Successful: {sum(1 for r in results if r['error_message'] is None)}")
print(f"  Errors: {sum(1 for r in results if r['error_message'] is not None)}")

In [0]:
# Cell 7: Build DataFrame & Save

from pyspark.sql.types import StructType, StructField, StringType, BooleanType, FloatType

output_schema = StructType([
    StructField("input_path", StringType(), True),
    StructField("is_loaded_ok", BooleanType(), True),
    StructField("is_text_detected", BooleanType(), True),
    StructField("is_pii_detected", BooleanType(), True),
    StructField("confidence_score", FloatType(), True),
    StructField("detected_text_summary", StringType(), True),
    StructField("llm_raw_result", StringType(), True),
    StructField("error_message", StringType(), True),
])

results_df = spark.createDataFrame(results, schema=output_schema)

output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('input_folder')).replace('/', '')}_batch_output.parquet"
results_df.write.mode("overwrite").parquet(output_path)
print(f"Results saved to: {output_path}")

display(results_df)